In [ ]:
import duckdb
import os

CARPETA_RESULTADOS = r"C:\SERCOP_FINAL\03_resultados"
RUTA_BASE_MODELADO = os.path.join(CARPETA_RESULTADOS, "base_modelado_v1.parquet").replace("\\", "/")

con = duckdb.connect(database=':memory:')

print(f"Base de modelado: {RUTA_BASE_MODELADO}")
n_filas = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_BASE_MODELADO}')").fetchone()[0]
print(f"Filas: {n_filas:,}")

In [ ]:
# --- Antes de normalizar: cuántas categorías distintas hay en crudo ---
n_categorias_crudas = con.execute(f"""
    SELECT COUNT(DISTINCT tipo_contratacion) AS n
    FROM read_parquet('{RUTA_BASE_MODELADO}')
""").fetchone()[0]
print(f"Categorías distintas de tipo_contratacion SIN normalizar: {n_categorias_crudas:,}")

In [ ]:
# --- Normalización: todo lo que sigue a "en el convenio" se recorta ---
query_normalizado = f"""
    SELECT
        *,
        CASE
            WHEN tipo_contratacion LIKE '%en el convenio%'
                THEN TRIM(SPLIT_PART(tipo_contratacion, 'en el convenio', 1))
            ELSE TRIM(tipo_contratacion)
        END AS tipo_contratacion_canonico
    FROM read_parquet('{RUTA_BASE_MODELADO}')
"""

n_categorias_canonicas = con.execute(f"""
    SELECT COUNT(DISTINCT tipo_contratacion_canonico) AS n
    FROM ({query_normalizado})
""").fetchone()[0]
print(f"Categorías distintas de tipo_contratacion_canonico DESPUÉS de normalizar: {n_categorias_canonicas:,}")

# --- Distribución final ---
distribucion_canonica = con.execute(f"""
    SELECT tipo_contratacion_canonico, COUNT(*) AS n
    FROM ({query_normalizado})
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print("\nDistribución completa de tipo_contratacion_canonico:")
print(distribucion_canonica.to_string(index=False))

In [ ]:
# --- Guardar tipo_contratacion_canonico como columna definitiva ---
RUTA_V2 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v2.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_normalizado})
    TO '{RUTA_V2}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v2 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V2}')").fetchone()[0]
print(f"base_modelado_v2.parquet: {n_v2:,} filas")
assert n_v2 == n_filas, "Descuadre al guardar la versión con tipo_contratacion_canonico"
print("OK: coincide con la base de modelado original.")

In [ ]:
# --- Diagnóstico: ¿el nulo de n_oferentes es estructural según el tipo de contratación? ---
nulos_por_tipo = con.execute(f"""
    SELECT
        tipo_contratacion_canonico,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(CASE WHEN n_oferentes IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_n_oferentes_nulo,
        ROUND(100.0 * SUM(CASE WHEN fecha_inicio_tender IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_fecha_inicio_nulo
    FROM read_parquet('{RUTA_V2}')
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print(nulos_por_tipo.to_string(index=False))

In [ ]:
# --- Construir bandera estructural: tiene_periodo_competitivo ---
query_con_bandera = f"""
    SELECT
        *,
        (fecha_inicio_tender IS NOT NULL) AS tiene_periodo_competitivo
    FROM read_parquet('{RUTA_V2}')
"""

resumen_bandera = con.execute(f"""
    SELECT
        tiene_periodo_competitivo,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / {n_v2}, 1) AS pct_del_total
    FROM ({query_con_bandera})
    GROUP BY 1
""").fetchdf()
print(resumen_bandera.to_string(index=False))

In [ ]:
# --- Guardar como base_modelado_v3.parquet ---
RUTA_V3 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v3.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_con_bandera})
    TO '{RUTA_V3}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v3 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V3}')").fetchone()[0]
assert n_v3 == n_v2, "Descuadre al agregar tiene_periodo_competitivo"
print(f"base_modelado_v3.parquet: {n_v3:,} filas — OK")

In [ ]:
RUTA_V3 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v3.parquet").replace("\\", "/")

# --- Diagnóstico: ¿cuántos procesos tienen cada combinación de monto_tender/monto_award? ---
combinaciones_monto = con.execute(f"""
    SELECT
        (monto_tender IS NOT NULL) AS tiene_monto_tender,
        (monto_award IS NOT NULL) AS tiene_monto_award,
        COUNT(*) AS n
    FROM read_parquet('{RUTA_V3}')
    GROUP BY 1, 2
    ORDER BY n DESC
""").fetchdf()
print(combinaciones_monto.to_string(index=False))

In [ ]:
# --- Construir monto_final: prioriza el monto adjudicado (award) sobre el estimado (tender) ---
query_con_monto_final = f"""
    SELECT
        *,
        COALESCE(monto_award, monto_tender) AS monto_final,
        CASE
            WHEN monto_award IS NOT NULL THEN 'award'
            WHEN monto_tender IS NOT NULL THEN 'tender'
            ELSE 'sin_dato'
        END AS fuente_monto
    FROM read_parquet('{RUTA_V3}')
"""

resumen_fuente = con.execute(f"""
    SELECT fuente_monto, COUNT(*) AS n, ROUND(100.0*COUNT(*)/{n_v3}, 2) AS pct
    FROM ({query_con_monto_final})
    GROUP BY 1
    ORDER BY n DESC
""").fetchdf()
print(resumen_fuente.to_string(index=False))

In [ ]:
# --- Guardar monto_final / fuente_monto ---
RUTA_V4 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v4.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_con_monto_final})
    TO '{RUTA_V4}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v4 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V4}')").fetchone()[0]
assert n_v4 == n_v3, "Descuadre al agregar monto_final/fuente_monto"
print(f"base_modelado_v4.parquet: {n_v4:,} filas — OK")

In [ ]:
# --- Separar los procesos sin ningún monto (no se pueden usar para indicadores de precio) ---
RUTA_SIN_MONTO = os.path.join(CARPETA_RESULTADOS, "procesos_sin_monto.parquet").replace("\\", "/")
RUTA_V5 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v5.parquet").replace("\\", "/")

con.execute(f"""
    COPY (SELECT * FROM read_parquet('{RUTA_V4}') WHERE fuente_monto = 'sin_dato')
    TO '{RUTA_SIN_MONTO}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute(f"""
    COPY (SELECT * FROM read_parquet('{RUTA_V4}') WHERE fuente_monto != 'sin_dato')
    TO '{RUTA_V5}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_sin_monto = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_SIN_MONTO}')").fetchone()[0]
n_v5 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V5}')").fetchone()[0]

print(f"Sin monto (excluidos de indicadores de precio): {n_sin_monto:,}")
print(f"Base de modelado v5 (con monto):                {n_v5:,}")
print(f"Suma:                                            {n_sin_monto + n_v5:,}")
assert n_sin_monto + n_v5 == n_v4
print("OK: reconcilia con base_modelado_v4.")

In [ ]:
# --- Paso A: mediana de monto_final por tipo (agregación simple) ---
mediana_simple = con.execute(f"""
    SELECT tipo_contratacion_canonico, COUNT(*) AS n, MEDIAN(monto_final) AS mediana_monto
    FROM read_parquet('{RUTA_V5}')
    GROUP BY 1
""").fetchdf()
print(mediana_simple.to_string(index=False))

In [ ]:
# --- Paso B: usando esa mediana, calcular el MAD (mediana de |x - mediana|) por tipo ---
query_medianas = f"""
    SELECT tipo_contratacion_canonico, MEDIAN(monto_final) AS mediana_monto
    FROM read_parquet('{RUTA_V5}')
    GROUP BY 1
"""

query_mad = f"""
    WITH medianas AS ({query_medianas}),
    con_mediana AS (
        SELECT b.*, m.mediana_monto
        FROM read_parquet('{RUTA_V5}') b
        JOIN medianas m USING (tipo_contratacion_canonico)
    )
    SELECT tipo_contratacion_canonico, MEDIAN(mediana_monto) AS mediana_monto,
           MEDIAN(ABS(monto_final - mediana_monto)) AS mad_monto
    FROM con_mediana
    GROUP BY 1
"""

mad_por_tipo = con.execute(query_mad).fetchdf()
print(mad_por_tipo.to_string(index=False))

In [ ]:
# --- Paso C: construir desviacion_precio_z usando esas medianas/MAD por tipo ---
query_con_desviacion = f"""
    WITH stats AS ({query_mad}),
    con_stats AS (
        SELECT b.*, s.mediana_monto AS mediana_monto_tipo, s.mad_monto AS mad_monto_tipo
        FROM read_parquet('{RUTA_V5}') b
        JOIN stats s USING (tipo_contratacion_canonico)
    )
    SELECT
        *,
        0.6745 * (monto_final - mediana_monto_tipo) / NULLIF(mad_monto_tipo, 0) AS desviacion_precio_z
    FROM con_stats
"""

n_check = con.execute(f"SELECT COUNT(*) FROM ({query_con_desviacion})").fetchone()[0]
print(f"Filas resultantes: {n_check:,}")

diag_z = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN desviacion_precio_z IS NULL THEN 1 ELSE 0 END) AS n_nulo,
        MIN(desviacion_precio_z) AS min_z,
        MAX(desviacion_precio_z) AS max_z,
        MEDIAN(desviacion_precio_z) AS mediana_z,
        SUM(CASE WHEN ABS(desviacion_precio_z) > 3.5 THEN 1 ELSE 0 END) AS n_atipicos_umbral_3_5
    FROM ({query_con_desviacion})
""").fetchdf()
print(diag_z.to_string(index=False))

In [ ]:
# --- Diagnóstico: ¿hay montos en cero o negativos? ---
diag_monto = con.execute(f"""
    SELECT
        MIN(monto_final) AS min_monto,
        SUM(CASE WHEN monto_final = 0 THEN 1 ELSE 0 END) AS n_monto_cero,
        SUM(CASE WHEN monto_final < 0 THEN 1 ELSE 0 END) AS n_monto_negativo
    FROM read_parquet('{RUTA_V5}')
""").fetchdf()
print(diag_monto.to_string(index=False))

In [ ]:
# --- Query completa: log del monto, mediana/MAD en escala log, por tipo (nulo incluido como categoría) ---

query_base_log = f"""
    SELECT
        *,
        COALESCE(tipo_contratacion_canonico, 'SIN_TIPO_CONTRATACION') AS tipo_contratacion_grp,
        (monto_final = 0) AS monto_cero,
        CASE WHEN monto_final > 0 THEN LN(monto_final) ELSE NULL END AS log_monto
    FROM read_parquet('{RUTA_V5}')
"""

query_medianas_log = f"""
    SELECT tipo_contratacion_grp, MEDIAN(log_monto) AS mediana_log
    FROM ({query_base_log})
    WHERE log_monto IS NOT NULL
    GROUP BY 1
"""

query_mad_log = f"""
    WITH medianas AS ({query_medianas_log}),
    con_mediana AS (
        SELECT b.*, m.mediana_log
        FROM ({query_base_log}) b
        JOIN medianas m USING (tipo_contratacion_grp)
    )
    SELECT tipo_contratacion_grp, MEDIAN(mediana_log) AS mediana_log,
           MEDIAN(ABS(log_monto - mediana_log)) AS mad_log
    FROM con_mediana
    WHERE log_monto IS NOT NULL
    GROUP BY 1
"""

query_final_z = f"""
    WITH stats AS ({query_mad_log}),
    con_stats AS (
        SELECT b.*, s.mediana_log, s.mad_log
        FROM ({query_base_log}) b
        JOIN stats s USING (tipo_contratacion_grp)
    )
    SELECT
        *,
        0.6745 * (log_monto - mediana_log) / NULLIF(mad_log, 0) AS desviacion_precio_log_z
    FROM con_stats
"""

n_check2 = con.execute(f"SELECT COUNT(*) FROM ({query_final_z})").fetchone()[0]
print(f"Filas resultantes: {n_check2:,}  (debería ser 2,736,387)")

diag_z2 = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN monto_cero THEN 1 ELSE 0 END) AS n_monto_cero,
        SUM(CASE WHEN desviacion_precio_log_z IS NULL AND NOT monto_cero THEN 1 ELSE 0 END) AS n_nulo_inesperado,
        MIN(desviacion_precio_log_z) AS min_z,
        MAX(desviacion_precio_log_z) AS max_z,
        MEDIAN(desviacion_precio_log_z) AS mediana_z,
        SUM(CASE WHEN ABS(desviacion_precio_log_z) > 3.5 THEN 1 ELSE 0 END) AS n_atipicos_umbral_3_5
    FROM ({query_final_z})
""").fetchdf()
print(diag_z2.to_string(index=False))

In [ ]:
RUTA_V6 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v6.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_final_z})
    TO '{RUTA_V6}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v6 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V6}')").fetchone()[0]
assert n_v6 == n_check2
print(f"base_modelado_v6.parquet: {n_v6:,} filas — OK")
print("\nColumnas nuevas en esta etapa: tipo_contratacion_grp, monto_cero, log_monto, mediana_log, mad_log, desviacion_precio_log_z")

In [ ]:
# --- Fragmentación de contratos: conteo y suma acumulada por par entidad-proveedor
#     en una ventana móvil de 30 días (usando fecha_release como proxy de fecha del proceso) ---

query_fragmentacion = f"""
    SELECT
        *,
        CASE WHEN entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL THEN
            COUNT(*) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            )
        ELSE NULL END AS n_contratos_30d_par,
        CASE WHEN entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL THEN
            SUM(monto_final) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            )
        ELSE NULL END AS monto_acumulado_30d_par
    FROM read_parquet('{RUTA_V6}')
"""

n_check3 = con.execute(f"SELECT COUNT(*) FROM ({query_fragmentacion})").fetchone()[0]
print(f"Filas resultantes: {n_check3:,}  (debería ser 2,736,387)")

In [ ]:
# --- Diagnóstico: distribución de n_contratos_30d_par ---
diag_frag = con.execute(f"""
    SELECT n_contratos_30d_par, COUNT(*) AS n
    FROM ({query_fragmentacion})
    WHERE n_contratos_30d_par IS NOT NULL
    GROUP BY 1
    ORDER BY 1
    LIMIT 15
""").fetchdf()
print(diag_frag.to_string(index=False))

resumen_frag = con.execute(f"""
    SELECT
        COUNT(*) AS n_con_par_ruc,
        SUM(CASE WHEN n_contratos_30d_par >= 3 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_3mas,
        SUM(CASE WHEN n_contratos_30d_par >= 5 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_5mas
    FROM ({query_fragmentacion})
    WHERE n_contratos_30d_par IS NOT NULL
""").fetchdf()
print("\nResumen:")
print(resumen_frag.to_string(index=False))

In [ ]:
# --- ¿Qué tipo de contratación domina los casos "n_contratos_30d_par >= 3"? ---
diag_por_tipo = con.execute(f"""
    SELECT tipo_contratacion_grp, COUNT(*) AS n
    FROM ({query_fragmentacion})
    WHERE n_contratos_30d_par >= 3
    GROUP BY 1
    ORDER BY n DESC
    LIMIT 10
""").fetchdf()
print(diag_por_tipo.to_string(index=False))

In [ ]:
# --- Fragmentación recalculada: SOLO sobre procesos que NO son catálogo electrónico ---

query_frag_v2 = f"""
    WITH base_no_catalogo AS (
        SELECT *
        FROM ({query_fragmentacion})
        WHERE tipo_contratacion_grp NOT LIKE 'Catálogo electrónico%'
          AND entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL
    ),
    con_ventana AS (
        SELECT
            ocid,
            COUNT(*) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            ) AS n_contratos_30d_no_catalogo,
            SUM(monto_final) OVER (
                PARTITION BY entidad_ruc, proveedor_ruc
                ORDER BY fecha_release
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            ) AS monto_acumulado_30d_no_catalogo
        FROM base_no_catalogo
    )
    SELECT
        b.* EXCLUDE (n_contratos_30d_par, monto_acumulado_30d_par),
        v.n_contratos_30d_no_catalogo,
        v.monto_acumulado_30d_no_catalogo
    FROM ({query_fragmentacion}) b
    LEFT JOIN con_ventana v USING (ocid)
"""

n_check4 = con.execute(f"SELECT COUNT(*) FROM ({query_frag_v2})").fetchone()[0]
print(f"Filas resultantes: {n_check4:,}  (debería ser 2,736,387)")

In [ ]:
# --- Diagnóstico del indicador refinado ---
resumen_frag_v2 = con.execute(f"""
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN n_contratos_30d_no_catalogo IS NOT NULL THEN 1 ELSE 0 END) AS n_elegibles,
        SUM(CASE WHEN n_contratos_30d_no_catalogo >= 3 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_3mas,
        SUM(CASE WHEN n_contratos_30d_no_catalogo >= 5 THEN 1 ELSE 0 END) AS n_posible_fragmentacion_5mas
    FROM ({query_frag_v2})
""").fetchdf()
print(resumen_frag_v2.to_string(index=False))

In [ ]:
RUTA_V7 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v7.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_frag_v2})
    TO '{RUTA_V7}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v7 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V7}')").fetchone()[0]
assert n_v7 == n_check4
print(f"base_modelado_v7.parquet: {n_v7:,} filas — OK")

In [ ]:
# --- Indicador de proveedor único: SOLO tiene sentido donde hubo periodo competitivo ---

resumen_n_oferentes = con.execute(f"""
    SELECT
        tiene_periodo_competitivo,
        COUNT(*) AS n,
        SUM(CASE WHEN n_oferentes IS NULL THEN 1 ELSE 0 END) AS n_nulo,
        MIN(n_oferentes) AS min_oferentes,
        MAX(n_oferentes) AS max_oferentes,
        SUM(CASE WHEN n_oferentes = 1 THEN 1 ELSE 0 END) AS n_proveedor_unico
    FROM read_parquet('{RUTA_V7}')
    GROUP BY 1
""").fetchdf()
print(resumen_n_oferentes.to_string(index=False))

In [ ]:
# --- Construir la columna definitiva proveedor_unico ---
query_proveedor_unico = f"""
    SELECT
        *,
        CASE
            WHEN tiene_periodo_competitivo THEN (n_oferentes = 1)
            ELSE NULL
        END AS proveedor_unico
    FROM read_parquet('{RUTA_V7}')
"""

RUTA_V8 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v8.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_proveedor_unico})
    TO '{RUTA_V8}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v8 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V8}')").fetchone()[0]
print(f"base_modelado_v8.parquet: {n_v8:,} filas — OK" if n_v8 == 2736387 else f"OJO: {n_v8:,} filas, revisar")

resumen_final = con.execute(f"""
    SELECT
        proveedor_unico,
        COUNT(*) AS n
    FROM read_parquet('{RUTA_V8}')
    GROUP BY 1
""").fetchdf()
print(resumen_final.to_string(index=False))

In [ ]:
# --- Concentración entidad-proveedor: % del gasto total de la entidad que va a cada proveedor ---

query_concentracion = f"""
    SELECT
        *,
        SUM(monto_final) OVER (PARTITION BY entidad_ruc, proveedor_ruc) AS monto_total_historico_par,
        SUM(monto_final) OVER (PARTITION BY entidad_ruc) AS monto_total_historico_entidad,
        COUNT(DISTINCT proveedor_ruc) OVER (PARTITION BY entidad_ruc) AS n_proveedores_distintos_entidad
    FROM read_parquet('{RUTA_V8}')
"""

query_concentracion_final = f"""
    SELECT
        *,
        CASE WHEN entidad_ruc IS NOT NULL AND proveedor_ruc IS NOT NULL THEN
            monto_total_historico_par / NULLIF(monto_total_historico_entidad, 0)
        ELSE NULL END AS concentracion_entidad_proveedor
    FROM ({query_concentracion})
"""

n_check5 = con.execute(f"SELECT COUNT(*) FROM ({query_concentracion_final})").fetchone()[0]
print(f"Filas resultantes: {n_check5:,}  (debería ser 2,736,387)")

In [ ]:
# --- Diagnóstico: distribución de la concentración y de n_proveedores_distintos_entidad ---
diag_concentracion = con.execute(f"""
    SELECT
        COUNT(*) AS n,
        SUM(CASE WHEN concentracion_entidad_proveedor IS NULL THEN 1 ELSE 0 END) AS n_nulo,
        MIN(n_proveedores_distintos_entidad) AS min_proveedores_entidad,
        MEDIAN(n_proveedores_distintos_entidad) AS mediana_proveedores_entidad,
        MAX(n_proveedores_distintos_entidad) AS max_proveedores_entidad,
        MEDIAN(concentracion_entidad_proveedor) AS mediana_concentracion,
        SUM(CASE WHEN concentracion_entidad_proveedor >= 0.5 THEN 1 ELSE 0 END) AS n_concentracion_50mas,
        SUM(CASE WHEN concentracion_entidad_proveedor >= 0.8 THEN 1 ELSE 0 END) AS n_concentracion_80mas
    FROM ({query_concentracion_final})
""").fetchdf()
print(diag_concentracion.to_string(index=False))

In [ ]:
RUTA_V9 = os.path.join(CARPETA_RESULTADOS, "base_modelado_v9.parquet").replace("\\", "/")

con.execute(f"""
    COPY ({query_concentracion_final})
    TO '{RUTA_V9}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

n_v9 = con.execute(f"SELECT COUNT(*) FROM read_parquet('{RUTA_V9}')").fetchone()[0]
print(f"base_modelado_v9.parquet: {n_v9:,} filas — OK" if n_v9 == 2736387 else f"OJO: {n_v9:,} filas, revisar")

cols_v9 = con.execute(f"SELECT * FROM read_parquet('{RUTA_V9}') LIMIT 0").df().columns.tolist()
print(f"\nTotal de columnas: {len(cols_v9)}")
print(cols_v9)